# 0916 13일차

## 1. Flatten

Conv 층의 출력을 한 줄로 펴서 `Dense`에 넘기는 층

```
(N, 20, 20, 16)  →  (N, 20 * 20 * 16)  →  (N, 6400)
```

### 1-1. Flatten이 필요한 이유

1. Conv 출력은 샘플 하나가 3차원 `(20, 20, 16)`이라 정답과 모양이 맞지 않음
   - `Dense`는 마지막 축에만 적용되므로, 펴지 않고 `Dense(10)`을 붙이면 출력이 `(20, 20, 10)`이 됨
   - 정답 `y`는 원핫이라 `(10,)` → 손실 함수가 자리마다 비교할 수 없음
2. Conv는 부분 영역만 보므로(12일차 §4-2), 뽑아낸 특징을 한자리에 모아 판단하는 `Dense`가 필요함

### 1-2. Flatten의 동작

1. 값도 순서도 바뀌지 않고 표현만 달라짐 → reshape와 같은 동작 (2일차 §3)
2. 배치 축 `N`은 그대로 두고 나머지 축을 하나로 합침 → `20 × 20 × 16 = 6400`
3. 계산하는 가중치가 없어서 파라미터는 0개

```python
model.add(Conv2D(16, (2, 2), activation='relu'))   # (N, 20, 20, 16)
model.add(Flatten())                               # (N, 6400)
model.add(Dense(units=32, activation='relu'))      # (N, 32)   6400개를 32개로 모음
model.add(Dense(10, activation='softmax'))         # (N, 10)
```

## 2. Conv2D의 filters

CNN에서 출력 채널 수는 `filters`가 정함 (12일차 §4-2)

### 2-1. 인자 쓰는 법

1. `Conv2D(32, (2, 2))` : 위치로 넘김. 첫 번째가 `filters`, 두 번째가 `kernel_size`
2. `Conv2D(filters=32, kernel_size=(2, 2))` : 이름으로 넘김

- 둘은 같은 코드임. `Dense(units=32)`와 `Dense(32)`가 같은 것과 마찬가지

## 3. padding

입력의 가장자리에 값을 채워 출력 크기를 조절하는 옵션

### 3-1. 패딩이 필요한 이유

1. 가장자리 정보가 적게 반영됨
   - 3×3 필터로 훑으면 픽셀이 필터에 들어가는 횟수가 위치마다 다름
   - 모서리는 1번, 변은 3번, 안쪽은 9번 → 가장자리에 중요한 정보가 있으면 덜 학습됨
2. 층을 지날수록 크기가 줄어듦
   - 한 층마다 `kernel - 1`씩 줄어 층을 깊게 쌓을 수 없음

### 3-2. padding의 종류

1. `'valid'` (기본값) : 채우지 않음 → 출력 = 입력 - kernel + 1
2. `'same'` : 출력이 입력과 같아지도록 바깥을 0으로 두름

**0으로 채우는 이유**
- 0은 곱해도 더해지는 값이 없어서, 없는 픽셀이 계산 결과를 바꾸지 않음

`(10, 10, 1)` 입력에 `Conv2D(10, (2, 2))`를 적용한 결과

| 설정 | 출력 | 파라미터 |
|---|---|---|
| `padding='valid'` | `(9, 9, 10)` | 50 |
| `padding='same'` | `(10, 10, 10)` | 50 |

- 패딩은 출력 크기만 바꾸고 파라미터 수는 바꾸지 않음 → 가중치는 필터 크기로 정해지기 때문

## 4. strides

필터를 몇 칸씩 옮길지 정하는 옵션 (기본값 `1`)

**쓰는 이유**
- 출력 크기를 줄여 연산량을 줄임. 2를 주면 가로·세로가 절반이 됨

### 4-1. 출력 크기 계산

1. `padding='valid'` : `(입력 - kernel) / strides + 1`, 나머지는 버림
2. `padding='same'` : `입력 / strides`, 올림

`(10, 10, 1)` 입력에 `Conv2D(10, (2, 2))`를 적용한 결과

| 설정 | 출력 |
|---|---|
| 기본값 (strides=1, valid) | `(9, 9, 10)` |
| `strides=2` | `(5, 5, 10)` |
| `padding='same'` | `(10, 10, 10)` |
| `padding='same', strides=2` | `(5, 5, 10)` |

- `padding='same'`으로 크기를 유지해도 `strides=2`가 먼저 크기를 절반으로 줄임

```python
model.add(Conv2D(10, (2, 2), input_shape=(10, 10, 1),
                 strides=2, padding='same'))     # (5, 5, 10)
model.add(Conv2D(filters=9, kernel_size=(3, 3),
                 strides=2, padding='same'))     # (3, 3, 9)
```

### 4-2. 주의) 건너뛴 자리는 보지 않음

1. strides가 1일 때는 잘린 영역끼리 겹치는데, 2가 되면 겹치는 부분이 사라져 정보가 빠짐
2. 그래서 가급적 권하지 않는 설정임
3. 크기를 줄이는 것이 목적이면 보통 `MaxPooling2D`처럼 줄이는 전용 층을 씀 (§5)

## 5. MaxPooling

구역을 나눠 각 구역에서 가장 큰 값만 남기는 층

### 5-1. MaxPooling이 필요한 이유

1. 이미지 전체를 같은 비중으로 훑을 필요가 없음
   - 식별에 중요한 부분은 일부에 몰려 있음 → 중요한 곳의 값만 남겨 집중함
2. Conv 층만 쌓으면 크기가 `kernel - 1`씩 조금씩만 줄어 연산이 계속 큼
   - `10×10`을 2×2 필터로 두 번 훑어야 `8×8`이 됨
   - 2×2 구역으로 나눠 최댓값만 뽑으면 한 번에 `5×5`가 됨

### 5-2. 동작

`pool_size=(2, 2)`로 2×2 구역을 나누고 각 구역의 최댓값만 남김

```
[[ 1  2  3  4]        [[ 6  8]
 [ 5  6  7  8]   →     [14 16]]
 [ 9 10 11 12]
 [13 14 15 16]]
```

1. `strides`의 기본값이 `pool_size`와 같음 → 구역이 겹치지 않고 가로·세로가 정확히 절반이 됨
2. 계산하는 가중치가 없어서 파라미터는 0개

**최댓값을 고르는 것의 의미**
- Conv와 relu를 지난 값은 그 자리에 필터가 찾는 특징이 얼마나 있는지를 뜻함
- 최댓값만 남긴다는 것은 그 구역에서 특징이 가장 뚜렷한 곳의 값만 남긴다는 뜻

### 5-3. 쓰는 위치 - Conv2D 다음

1. `Conv2D`가 특징을 뽑고, `MaxPooling2D`가 그중 강한 값만 남기는 순서임
2. 특징을 뽑기 전에 줄이면 원본 픽셀을 그냥 버리는 것이 됨

```python
model.add(Conv2D(10, (3, 3), activation='relu', input_shape=(28, 28, 1)))   # (26, 26, 10)
model.add(MaxPooling2D())                                                   # (13, 13, 10)
model.add(Conv2D(20, (3, 3), activation='relu'))                            # (11, 11, 20)
model.add(MaxPooling2D())                                                   # (5, 5, 20)
```

### 5-4. strides=2와의 차이

| | 크기 | 값을 고르는 방식 |
|---|---|---|
| `Conv2D(strides=2)` | 절반 | 건너뛴 자리는 보지 않음 |
| `MaxPooling2D` | 절반 | 구역 전체를 본 뒤 최댓값만 남김 |

- 그래서 크기를 줄일 때는 보통 MaxPooling을 씀 (§4-2)

### 5-5. 효과

1. 연산량 감소 : 칸 수가 1/4로 줄고, `Flatten` 뒤 `Dense` 파라미터도 크게 줄어듦
2. 위치 변화에 덜 민감 : 숫자가 한두 칸 밀려도 최댓값은 그대로라 결과가 비슷함
3. 파라미터가 줄어 과적합도 덜함 → 성능이 올라가는 경우가 있음